# Experiment Performance Analysis

Visualizes pass/fail distributions across Langfuse dataset runs (experiments).

## Prerequisites
- Langfuse running (default: `http://localhost:3000`)
- `.env` at workspace root with `LANGFUSE_PUBLIC_KEY`, `LANGFUSE_SECRET_KEY`, and optionally `EVAL_LANGFUSE_BASE_URL`
- At least one eval run pushed to Langfuse via `uv run eval-run --langfuse`

## Usage
Change `DATASET_NAME` below to analyze a different skill's experiments.
Run names correspond to `--run-name` values passed to `eval-run` (e.g. `scaffold-baseline`, `v2-prompt`).

In [1]:
import os
from pathlib import Path

from dotenv import load_dotenv

# Load env vars FIRST, then override LANGFUSE_BASE_URL before importing langfuse
load_dotenv(Path("../.env"))
os.environ["LANGFUSE_BASE_URL"] = (
    os.environ.get("EVAL_LANGFUSE_BASE_URL", "").strip() or "http://localhost:3000"
)

import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from langfuse import Langfuse

# --- Configuration ---
DATASET_NAME = "evals/Meeting Summariser"
# ---------------------

In [2]:
langfuse = Langfuse()
print(f"Connected to Langfuse at {os.environ['LANGFUSE_BASE_URL']}")

Connected to Langfuse at http://localhost:3000


In [3]:
# Fetch all runs for the dataset
runs_response = langfuse.get_dataset_runs(dataset_name=DATASET_NAME)
runs = runs_response.data
print(f"Found {len(runs)} experiment run(s) for '{DATASET_NAME}'")
for r in runs:
    print(f"  - {r.name} (created: {r.created_at})")

Found 1 experiment run(s) for 'evals/Meeting Summariser'
  - 1.baseline (created: 2026-05-15 07:59:59.351000+00:00)


In [4]:
# Fetch verdict scores for each run
records = []

for run in runs:
    run_detail = langfuse.get_dataset_run(
        dataset_name=DATASET_NAME, run_name=run.name
    )
    for item in run_detail.dataset_run_items:
        scores = langfuse.api.scores.get_many(
            trace_id=item.trace_id, name="verdict", limit=1
        )
        verdict = None
        if scores.data:
            verdict = bool(scores.data[0].value)
        records.append({
            "run_name": run.name,
            "created_at": run.created_at,
            "trace_id": item.trace_id,
            "dataset_item_id": item.dataset_item_id,
            "verdict": verdict,
        })

df = pd.DataFrame(records)
print(f"Collected {len(df)} trace verdicts across {df['run_name'].nunique()} runs")
df.head()

Collected 8 trace verdicts across 1 runs


,run_name,created_at,trace_id,dataset_item_id,verdict
0,1.baseline,2026-05-15 07:59:59.351000+00:00,242cb368ce39572b4fbc03337e2985d1,37bb98d2-1b92-5b82-b95f-7ecfc1367fd5,False
1,1.baseline,2026-05-15 07:59:59.351000+00:00,b19f20825c433237abbe00bf02c4fb5b,ba51e19b-b378-5ad1-875b-615d15eb127c,True
2,1.baseline,2026-05-15 07:59:59.351000+00:00,ff077de973e3cad8a41b3c40a0e79235,ea58ad42-3fa2-5f86-a942-6651952440bf,True
3,1.baseline,2026-05-15 07:59:59.351000+00:00,ae66b812c13b77cd6723fb979ea098c7,62c785b1-23cb-52e0-a8ef-ee747ca93a0e,True
4,1.baseline,2026-05-15 07:59:59.351000+00:00,7502a54773055c496dadb98b148d711d,fae147eb-5959-57bf-91f9-18e632a75a3a,True


In [5]:
# Aggregate pass/fail per run
run_order = (
    df.groupby("run_name")["created_at"]
    .first()
    .sort_values()
    .index.tolist()
)

summary = (
    df.groupby("run_name")["verdict"]
    .agg(total="count", passed=lambda x: x.sum(), failed=lambda x: (~x).sum())
    .reindex(run_order)
)
summary["pass_rate"] = (summary["passed"] / summary["total"] * 100).round(1)
summary

,total,passed,failed,pass_rate
run_name,,,,
1.baseline,8,6,2,75.0


In [8]:
# Visualization: Pass Rate Trend + Pass/Fail Distribution
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Pass Rate by Experiment Run", "Pass/Fail Distribution"),
    horizontal_spacing=0.12,
)

# Left panel: Pass rate trend line
fig.add_trace(
    go.Scatter(
        x=summary.index,
        y=summary["pass_rate"],
        mode="lines+markers+text",
        text=[f"{v}%" for v in summary["pass_rate"]],
        textposition="top center",
        textfont=dict(color="#2e7d32", size=12, family="Arial Black"),
        line=dict(color="#4caf50", width=3),
        marker=dict(size=10, color="#2e7d32"),
        showlegend=False,
    ),
    row=1, col=1,
)

# Right panel: Grouped bar chart
fig.add_trace(
    go.Bar(
        x=summary.index,
        y=summary["passed"],
        name="Pass",
        marker_color="#8bc34a",
        text=summary["passed"].astype(int),
        textposition="outside",
    ),
    row=1, col=2,
)
fig.add_trace(
    go.Bar(
        x=summary.index,
        y=summary["failed"],
        name="Fail",
        marker_color="#ef9a9a",
        text=summary["failed"].astype(int),
        textposition="outside",
    ),
    row=1, col=2,
)

fig.update_yaxes(title_text="Pass Rate (%)", range=[0, 100], row=1, col=1)
fig.update_yaxes(title_text="Count", row=1, col=2)
fig.update_layout(
    title_text=f"{DATASET_NAME.split('/')[-1]} Performance Analysis",
    title_x=0.5,
    title_font_size=18,
    barmode="group",
    height=450,
    width=1100,
    legend=dict(orientation="h", yanchor="bottom", y=-0.2, xanchor="center", x=0.75),
    template="plotly_white",
)
fig.show()

In [7]:
# Per-Case Heatmap: identify which cases regress across experiments
if df["run_name"].nunique() > 1:
    pivot = df.pivot_table(
        index="dataset_item_id",
        columns="run_name",
        values="verdict",
        aggfunc="first",
    )
    pivot = pivot.reindex(columns=run_order)

    # Map item IDs to short labels
    item_labels = [f"item_{i+1}" for i in range(len(pivot))]

    heatmap = go.Figure(
        data=go.Heatmap(
            z=pivot.values.astype(float),
            x=pivot.columns.tolist(),
            y=item_labels,
            colorscale=[[0, "#ef9a9a"], [1, "#8bc34a"]],
            showscale=False,
            zmin=0,
            zmax=1,
            text=pivot.applymap(lambda v: "PASS" if v else "FAIL").values,
            texttemplate="%{text}",
            hovertemplate="Run: %{x}<br>Item: %{y}<br>Verdict: %{text}<extra></extra>",
        )
    )
    heatmap.update_layout(
        title="Per-Case Verdicts Across Experiments",
        title_x=0.5,
        xaxis_title="Experiment Run",
        yaxis_title="Dataset Item",
        height=max(300, len(pivot) * 35),
        width=900,
        template="plotly_white",
    )
    heatmap.show()
else:
    print("Heatmap requires at least 2 experiment runs for comparison.")

Heatmap requires at least 2 experiment runs for comparison.
